# Figure 6 New

Uniform, weighted-undirected E. coli, and directed E. coli mutation models, plus controlled lethal and neutral analyses.


## Setup


In [1]:
%load_ext autoreload
%autoreload 2

import os
import pickle
import subprocess
import sys
from pathlib import Path

import jax.random as jr
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import PercentFormatter

from slide.data_generation import (
    generate_empirical_decay_curves,
    nk_grid_pairs,
    ordered_unique_pairs,
    random_start,
    run_nk_start_averaged_diffusion,
)
from slide.direvo_functions import get_single_decay_rate, get_single_decay_rate_IK_v2
from slide.ruggedness_functions import get_dirichlet_metric, get_nk_l_o_shape
from slide.utils import (
    FIGURE_LABEL_SIZE,
    FIGURE_LEGEND_SIZE,
    FIGURE_TICK_SIZE,
    FIGURE_TITLE_SIZE,
    PANEL_LETTER_SIZE,
    get_figures_dir,
    get_processed_data_dir,
    get_raw_data_dir,
    load_pickle,
    save_pickle,
)
from slide_config import get_slide_data_dir

OVERWRITE_RAW_PKL: bool = False
OVERWRITE_PROCESSED_PKL: bool = True
PLOT_ONLY: bool = False
SAVE_FIGURES: bool = True
PANEL_DPI: int = 350
SAVE_TYPES: tuple[str, ...] = ("pdf", "png", "eps")

RAW_DATA_DIR = get_raw_data_dir()
PROCESSED_DATA_DIR = get_processed_data_dir()
FIGURES_DIR = get_figures_dir()
SLIDE_DATA_DIR = Path(get_slide_data_dir())
REPO_ROOT = Path.cwd()

MODEL_KEYS: tuple[str, ...] = (
    "nuc_uniform",
    "nuc_e_coli_weighted",
    "nuc_e_coli_directed",
)
MODEL_TITLES = {
    "nuc_uniform": "Uniform mutation",
    "nuc_e_coli_weighted": r"Weighted $\mathit{E.\ coli}$",
    "nuc_e_coli_directed": r"Directed $\mathit{E.\ coli}$",
}
LANDSCAPE_KEYS: tuple[str, ...] = ("gb1", "trpb", "tev", "pard3")
LANDSCAPE_NAMES: tuple[str, ...] = ("GB1", "TrpB", "TEV", "ParD3")
LANDSCAPE_COLORS: tuple[str, ...] = ("tab:orange", "tab:blue", "tab:green", "tab:red")
LANDSCAPE_MARKERS: tuple[str, ...] = ("o", "s", "^", "D")

ABC_PROCESSED_PATH = PROCESSED_DATA_DIR / "figure6_ecoli_nk_local_gmu_processed.pkl"
DF_PROCESSED_PATH = PROCESSED_DATA_DIR / "figure6_ecoli_sampling_75steps_processed.pkl"
ANALYTICS_PATH = PROCESSED_DATA_DIR / "figure6_ecoli_kernel_analytics_processed.pkl"
ABC_RAW_PATHS = {
    model: RAW_DATA_DIR / f"figure6_ecoli_nk_{model}_raw.pkl"
    for model in MODEL_KEYS
}

# Panels A-C: retained Figure S2 design.
ABC_N_VALUES: tuple[int, ...] = (10, 14, 18, 23, 27, 32, 36, 41, 45, 50)
ABC_NUM_ALLELES: int = 4
ABC_NUM_K_VALUES_PER_N: int = 10
ABC_NUM_LANDSCAPES: int = 25
ABC_NUM_STARTS: int = 25
ABC_NUM_REPLICATES: int = 5
ABC_POPULATION_SIZE: int = 2_500
ABC_NUM_GENERATIONS: int = 25
ABC_TOTAL_MUTATION_RATE: float = 0.5

# Panels D-F and S3.
DF_NUM_GENERATIONS: int = 75
DF_TOTAL_MUTATION_RATE: float = 0.1
RANDOM_SEED: int = 42

# Existing panels G-H.
RAW_PATH = RAW_DATA_DIR / "figure6_new_nk_landscapes.pkl"
PROCESSED_PATH = PROCESSED_DATA_DIR / "figure6_new_lethal_neutral_rho2.pkl"
DECAY_RAW_PATH = RAW_DATA_DIR / "figure6_new_lethal_neutral_decay_curves.pkl"
DECAY_PROCESSED_PATH = PROCESSED_DATA_DIR / "figure6_new_lethal_neutral_decay_rho2.pkl"
N_SITES: int = 4
NUM_ALLELES: int = 20
K_VALUES: tuple[int, ...] = (0, 1, 2, 3)
FRACTIONS: tuple[float, ...] = (0.0, 0.05, 0.10, 0.20, 0.35, 0.50)
NUM_LANDSCAPES: int = 20
NUM_STARTS_PER_CLASS: int = 10
NUM_POPULATION_REPLICATES: int = 10
POPULATION_SIZE: int = 2_500
TOTAL_MUTATION_RATE: float = 0.1
NUM_DECAY_STEPS: int = 75
START_CLASSES: tuple[str, ...] = ("near_average", "high_fitness")
PERTURBATIONS: tuple[str, ...] = ("lethal", "neutral")

print(f"PLOT_ONLY={PLOT_ONLY}, OVERWRITE_RAW_PKL={OVERWRITE_RAW_PKL}, "
      f"OVERWRITE_PROCESSED_PKL={OVERWRITE_PROCESSED_PKL}")


def save_figure(fig: Figure, stem: str, *, bbox_inches: str = "tight") -> None:
    """Save one figure in every configured format.

    Parameters:
    - fig: Figure
        Figure to save.
    - stem: str
        Output filename stem.
    - bbox_inches: str
        Matplotlib bounding-box mode.

    Returns:
    - None
        Files are written below ``FIGURES_DIR``.
    """
    for suffix in SAVE_TYPES:
        destination = FIGURES_DIR / suffix
        destination.mkdir(parents=True, exist_ok=True)
        fig.savefig(destination / f"{stem}.{suffix}", dpi=PANEL_DPI, bbox_inches=bbox_inches)


def add_panel_letter(ax: Axes, letter: str) -> None:
    """Add a manuscript panel letter.

    Parameters:
    - ax: Axes
        Axis receiving the label.
    - letter: str
        Panel letter.

    Returns:
    - None
        The axis is modified in place.
    """
    ax.text(-0.14, 1.10, letter, transform=ax.transAxes,
            fontsize=PANEL_LETTER_SIZE, fontweight="bold", va="top", ha="left")


PLOT_ONLY=False, OVERWRITE_RAW_PKL=False, OVERWRITE_PROCESSED_PKL=True


## Mutation Kernels


In [2]:
def symmetric_sinkhorn_kernel(kernel: np.ndarray, tolerance: float = 1e-13) -> np.ndarray:
    """Create a symmetric doubly-stochastic kernel by diagonal scaling.

    Parameters:
    - kernel: np.ndarray
        Non-negative square base kernel.
    - tolerance: float
        Maximum permitted row-sum error.

    Returns:
    - np.ndarray
        Symmetric, doubly-stochastic kernel with the input zero pattern.
    """
    symmetric = 0.5 * (np.asarray(kernel, dtype=float) + np.asarray(kernel, dtype=float).T)
    scale = np.ones(symmetric.shape[0], dtype=float)
    for _ in range(100_000):
        row_sums = scale * (symmetric @ scale)
        if np.max(np.abs(row_sums - 1.0)) < tolerance:
            break
        scale *= np.sqrt(1.0 / row_sums)
    else:
        raise RuntimeError("Symmetric Sinkhorn scaling did not converge.")
    result = scale[:, None] * symmetric * scale[None, :]
    return result


def stationary_distribution(kernel: np.ndarray) -> np.ndarray:
    """Return the normalized stationary distribution of a row-stochastic kernel.

    Parameters:
    - kernel: np.ndarray
        Irreducible row-stochastic transition kernel.

    Returns:
    - np.ndarray
        Positive stationary probability vector.
    """
    eigenvalues, eigenvectors = np.linalg.eig(np.asarray(kernel, dtype=float).T)
    index = int(np.argmin(np.abs(eigenvalues - 1.0)))
    stationary = np.real(eigenvectors[:, index])
    if stationary.sum() < 0:
        stationary *= -1
    stationary /= stationary.sum()
    return stationary


uniform_kernel = (np.ones((4, 4)) - np.eye(4)) / 3.0
e_coli_directed_kernel = np.asarray(
    np.load(REPO_ROOT / "other_data" / "normed_e_coli_matrix.npy"), dtype=float
)
e_coli_weighted_kernel = symmetric_sinkhorn_kernel(e_coli_directed_kernel)
MUTATION_KERNELS = {
    "nuc_uniform": uniform_kernel,
    "nuc_e_coli_weighted": e_coli_weighted_kernel,
    "nuc_e_coli_directed": e_coli_directed_kernel,
}

for model, kernel in MUTATION_KERNELS.items():
    if kernel.shape != (4, 4) or np.any(kernel < 0):
        raise ValueError(f"Invalid mutation kernel for {model}.")
    if not np.allclose(kernel.sum(axis=1), 1.0, atol=1e-12):
        raise ValueError(f"Mutation kernel {model} is not row-stochastic.")
    adjacency = kernel > 0
    reachability = np.eye(4, dtype=bool)
    power = np.eye(4, dtype=bool)
    for _ in range(1, 4):
        power = (power.astype(int) @ adjacency.astype(int)) > 0
        reachability |= power
    if not np.all(reachability):
        raise ValueError(f"Mutation kernel {model} is not irreducible.")
if not np.allclose(e_coli_weighted_kernel, e_coli_weighted_kernel.T, atol=1e-12):
    raise ValueError("Weighted E. coli kernel is not symmetric.")
if not np.allclose(e_coli_weighted_kernel.sum(axis=0), 1.0, atol=1e-12):
    raise ValueError("Weighted E. coli kernel is not column-stochastic.")
if not np.allclose(np.diag(e_coli_weighted_kernel), 0.0, atol=1e-14):
    raise ValueError("Weighted E. coli kernel does not preserve the zero diagonal.")

STATIONARY_DISTRIBUTIONS = {
    model: stationary_distribution(kernel)
    for model, kernel in MUTATION_KERNELS.items()
}


## Raw Synthetic NK Products for A-C


In [3]:
raw_pairs = nk_grid_pairs((10, 50), ABC_NUM_K_VALUES_PER_N, K_start=0)
abc_nk_pairs = ordered_unique_pairs(raw_pairs)
if len(abc_nk_pairs) != 100:
    raise AssertionError("Expected 100 Figure 6 A-C NK pairs.")

abc_raw_payloads: dict[str, dict[str, object]] = {}
if not PLOT_ONLY:
    pair_keys = jr.split(jr.PRNGKey(RANDOM_SEED), len(abc_nk_pairs))
    for model in MODEL_KEYS:
        path = ABC_RAW_PATHS[model]
        if path.exists() and not OVERWRITE_RAW_PKL:
            abc_raw_payloads[model] = load_pickle(path)
            continue
        trajectories = np.empty(
            (len(abc_nk_pairs), ABC_NUM_LANDSCAPES, ABC_NUM_STARTS,
             ABC_NUM_REPLICATES, ABC_NUM_GENERATIONS), dtype=np.float32,
        )
        starts_padded = np.full(
            (len(abc_nk_pairs), ABC_NUM_LANDSCAPES, ABC_NUM_STARTS, max(ABC_N_VALUES)),
            -1, dtype=np.int8,
        )
        landscape_keys_saved = np.empty(
            (len(abc_nk_pairs), ABC_NUM_LANDSCAPES, 2), dtype=np.uint32,
        )
        for pair_index, (pair_key, pair) in enumerate(zip(pair_keys, abc_nk_pairs, strict=True)):
            n_sites, k_value = pair
            landscape_keys = jr.split(pair_key, ABC_NUM_LANDSCAPES)
            for landscape_index, landscape_key in enumerate(landscape_keys):
                start_keys = jr.split(
                    jr.fold_in(landscape_key, 100_000 + landscape_index), ABC_NUM_STARTS
                )
                starts = np.asarray([
                    random_start(key, n_sites=n_sites, num_alleles=ABC_NUM_ALLELES)
                    for key in start_keys
                ])
                starts_padded[pair_index, landscape_index, :, :n_sites] = starts
                landscape_keys_saved[pair_index, landscape_index] = np.asarray(landscape_key)
                trajectories[pair_index, landscape_index] = run_nk_start_averaged_diffusion(
                    rng_key=landscape_key,
                    trajectory_rng_key=landscape_key,
                    n_sites=n_sites,
                    k=k_value,
                    num_alleles=ABC_NUM_ALLELES,
                    starts=starts,
                    popsize=ABC_POPULATION_SIZE,
                    mutation_rate_per_site=ABC_TOTAL_MUTATION_RATE / n_sites,
                    num_reps_per_start=ABC_NUM_REPLICATES,
                    num_steps=ABC_NUM_GENERATIONS,
                    mutation_matrix=MUTATION_KERNELS[model],
                    return_replicates=True,
                )
        payload = {
            "data": {
                "fitness_trajectories": trajectories,
                "start_coordinates_padded": starts_padded,
                "landscape_keys": landscape_keys_saved,
                "mutation_kernel": MUTATION_KERNELS[model],
            },
            "params": {
                "model": model, "A": ABC_NUM_ALLELES, "nk_pairs": abc_nk_pairs,
                "num_landscapes": ABC_NUM_LANDSCAPES, "num_starts": ABC_NUM_STARTS,
                "num_population_replicates": ABC_NUM_REPLICATES,
                "population_size": ABC_POPULATION_SIZE, "M": ABC_NUM_GENERATIONS,
                "total_mutation_rate": ABC_TOTAL_MUTATION_RATE, "seed": RANDOM_SEED,
            },
            "metadata": {"paper_reference": "Figure 6A-C"},
        }
        save_pickle(payload, path)
        abc_raw_payloads[model] = payload
else:
    print("PLOT_ONLY=True: skipping Figure 6 A-C raw generation.")


## Processed Synthetic NK Products for A-C


In [ ]:
def process_abc_payload(raw_by_model: dict[str, dict[str, object]]) -> dict[str, object]:
    """Fit and bin local squared-fitness decay rates for panels A-C.

    Parameters:
    - raw_by_model: dict[str, dict[str, object]]
        Replicate-level raw trajectory payloads.

    Returns:
    - dict[str, object]
        Per-start fits and binned summaries.
    """
    output: dict[str, object] = {}
    for model in MODEL_KEYS:
        raw_payload = raw_by_model[model]
        trajectories = np.asarray(raw_payload["data"]["fitness_trajectories"], dtype=float)
        pairs = tuple((int(n), int(k)) for n, k in raw_payload["params"]["nk_pairs"])
        f_mu = trajectories.mean(axis=3)
        g_mu = np.square(f_mu)
        rho_local = np.full(g_mu.shape[:-1], np.nan, dtype=float)
        fit_success = np.zeros(g_mu.shape[:-1], dtype=bool)
        for index in np.ndindex(g_mu.shape[:-1]):
            try:
                rho_local[index] = float(get_single_decay_rate(
                    g_mu[index], mut=2.0 * ABC_TOTAL_MUTATION_RATE,
                    num_steps=ABC_NUM_GENERATIONS,
                )[0])
                fit_success[index] = True
            except (RuntimeError, ValueError, FloatingPointError):
                continue
        rho_nk = np.asarray([(k + 1) / n for n, k in pairs], dtype=float)
        repeated = np.broadcast_to(rho_nk[:, None, None], rho_local.shape)
        valid_rho = repeated[fit_success]
        valid_estimates = rho_local[fit_success]
        bin_edges = np.linspace(0.0, 1.0, 11)
        bin_indices = np.clip(np.digitize(valid_rho, bin_edges, right=True) - 1, 0, 9)
        binned_rho, binned_mean, binned_std, binned_counts = [], [], [], []
        for bin_index in range(10):
            selected = bin_indices == bin_index
            if np.any(selected):
                binned_rho.append(float(valid_rho[selected].mean()))
                binned_mean.append(float(valid_estimates[selected].mean()))
                binned_std.append(float(valid_estimates[selected].std()))
                binned_counts.append(int(selected.sum()))
        output[model] = {
            "rho_2_local": rho_local,
            "fit_success": fit_success,
            "rho_NK": rho_nk,
            "binned_rho_NK": np.asarray(binned_rho),
            "binned_mean": np.asarray(binned_mean),
            "binned_std": np.asarray(binned_std),
            "binned_counts": np.asarray(binned_counts),
        }
    return {
        "data": output,
        "params": {"models": MODEL_KEYS, "M": ABC_NUM_GENERATIONS,
                   "total_mutation_rate": ABC_TOTAL_MUTATION_RATE},
        "metadata": {"paper_reference": "Figure 6A-C"},
    }


if ABC_PROCESSED_PATH.exists() and (PLOT_ONLY or not OVERWRITE_PROCESSED_PKL):
    print(f"Loading pre-processed Figure 6 A-C payload from {ABC_PROCESSED_PATH}")
    figure6_abc_payload = load_pickle(ABC_PROCESSED_PATH)
elif PLOT_ONLY:
    raise FileNotFoundError(f"PLOT_ONLY=True requires {ABC_PROCESSED_PATH}")
else:
    print(f"Processing. Does not eist yet: {ABC_PROCESSED_PATH}")
    figure6_abc_payload = process_abc_payload(abc_raw_payloads)
    save_pickle(figure6_abc_payload, ABC_PROCESSED_PATH)


KeyboardInterrupt: 

## Raw Empirical Codon Products for D-F


In [5]:
required_df_raw_paths = {
    model: [
        SLIDE_DATA_DIR / f"decay_curves_{landscape}_{model}_m0.1_all_starts_75steps.pkl"
        for landscape in LANDSCAPE_KEYS
    ]
    for model in MODEL_KEYS
}

missing_df_raw_paths = [
    path for paths in required_df_raw_paths.values() for path in paths if not path.exists()
]
if not PLOT_ONLY and (missing_df_raw_paths or OVERWRITE_RAW_PKL):
    environment = os.environ.copy()
    environment.update({
        "SLIDE_NUM_STEPS": "75",
        "SLIDE_MUTATION_MODELS": ",".join(MODEL_KEYS),
        "SLIDE_OVERWRITE": "1" if OVERWRITE_RAW_PKL else "0",
    })
    subprocess.run(
        [sys.executable, "scripts/empirical_landscape_decay_curves_codon_fast.py"],
        cwd=REPO_ROOT, env=environment, check=True,
    )
    missing_df_raw_paths = [
        path for paths in required_df_raw_paths.values() for path in paths if not path.exists()
    ]
print(f"Missing Figure 6 D-F raw products: {len(missing_df_raw_paths)}")


Loading landscapes...
Loading mutation matrices...

Mutation model: nuc_uniform

  Landscape: GB1  (4 AA → 12 nt)


  0%|          | 0/320 [00:00<?, ?it/s]2026-07-05 15:39:09.636997: W external/tsl/tsl/framework/bfc_allocator.cc:482] Allocator (GPU_0_bfc) ran out of memory trying to allocate 4.80GiB (rounded to 5150009600)requested by op 
2026-07-05 15:39:09.637038: W external/tsl/tsl/framework/bfc_allocator.cc:494] *___________________________________________________________________________________________________
E0705 15:39:09.637175  980356 pjrt_stream_executor_client.cc:2809] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 5150009552 bytes.
BufferAssignment OOM Debugging.
BufferAssignment stats:
             parameter allocation:    23.4KiB
              constant allocation:   760.0KiB
        maybe_live_out allocation:    1.43MiB
     preallocated temp allocation:    4.80GiB
                 total allocation:    4.80GiB
Peak buffers:
	Buffer 1:
		Size: 3.49GiB
		XLA Label: fusion
		Shape: f32[75,500,10,2500]

	Buffer 2:
		Size: 572.20MiB
		Operator: op

CalledProcessError: Command '['/home/lady5906/miniconda3/envs/SLIDE_env/bin/python', 'scripts/empirical_landscape_decay_curves_codon_fast.py']' returned non-zero exit status 1.

## Processed Analytical Rates and Asymptotes for D-F


In [ ]:
from slide.direvo_functions import CODON_MAPPER


def build_nucleotide_landscape(landscape: np.ndarray) -> np.ndarray:
    """Expand an amino-acid landscape into nucleotide/codon space.

    Parameters:
    - landscape: np.ndarray
        Amino-acid landscape with one axis per residue.

    Returns:
    - np.ndarray
        Nucleotide landscape with three four-state axes per residue.
    """
    mapper = np.asarray(CODON_MAPPER, dtype=np.int16)
    buffered = np.pad(
        np.asarray(landscape, dtype=np.float64),
        [(0, 1)] * landscape.ndim,
        constant_values=float(np.min(landscape)),
    )
    indices = np.indices((4,) * (3 * landscape.ndim), dtype=np.int8)
    amino_acids = [
        mapper[indices[3 * site], indices[3 * site + 1], indices[3 * site + 2]]
        for site in range(landscape.ndim)
    ]
    return buffered[tuple(amino_acids)]


def apply_kernel_axis(values: np.ndarray, kernel: np.ndarray, axis: int) -> np.ndarray:
    """Apply one row-stochastic kernel to one function axis.

    Parameters:
    - values: np.ndarray
        Genotype-indexed function values.
    - kernel: np.ndarray
        Single-site row-stochastic kernel.
    - axis: int
        Axis receiving the kernel.

    Returns:
    - np.ndarray
        Kernel-transformed function values.
    """
    transformed = np.tensordot(kernel, values, axes=([1], [axis]))
    return np.moveaxis(transformed, 0, axis)


def stationary_average(values: np.ndarray, stationary: np.ndarray) -> float:
    """Compute the product-stationary average without forming a tensor product.

    Parameters:
    - values: np.ndarray
        Nucleotide-space landscape.
    - stationary: np.ndarray
        Single-site stationary distribution.

    Returns:
    - float
        Product-distribution weighted average.
    """
    contracted = np.asarray(values, dtype=np.float64)
    for _ in range(values.ndim):
        contracted = np.tensordot(stationary, contracted, axes=([0], [0]))
    return float(contracted)


def analytical_squared_decay(values: np.ndarray, kernel: np.ndarray, directed: bool) -> dict[str, object]:
    """Compute the analytical squared-decay metric for a product kernel.

    Parameters:
    - values: np.ndarray
        Nucleotide-space fitness landscape.
    - kernel: np.ndarray
        Single-site transition kernel.
    - directed: bool
        Whether to use the symmetrized directed Laplacian and stationary asymptote.

    Returns:
    - dict[str, object]
        Rate, asymptote, power term, and stationary distribution.
    """
    stationary = stationary_distribution(kernel)
    mean = stationary_average(values, stationary) if directed else float(values.mean())
    b_zero = float(values.size * mean * mean)
    denominator = float(np.vdot(values, values).real - b_zero)
    if denominator <= 0:
        raise ValueError("Squared-decay denominator must be positive.")
    laplacian_values = np.zeros_like(values, dtype=np.float64)
    for axis in range(values.ndim):
        forward = values - apply_kernel_axis(values, kernel, axis)
        if directed:
            reverse = values - apply_kernel_axis(values, kernel.T, axis)
            laplacian_values += 0.5 * (forward + reverse)
        else:
            laplacian_values += forward
    numerator = float(np.vdot(values, laplacian_values).real)
    rho_2 = numerator / (values.ndim * denominator)
    return {
        "rho_2": rho_2,
        "G_infinity": mean * mean,
        "b_0": b_zero,
        "stationary_distribution": stationary,
        "numerator_reduced": numerator,
    }


if ANALYTICS_PATH.exists() and (PLOT_ONLY or not OVERWRITE_PROCESSED_PKL):
    figure6_analytics = load_pickle(ANALYTICS_PATH)
elif PLOT_ONLY:
    raise FileNotFoundError(f"PLOT_ONLY=True requires {ANALYTICS_PATH}")
else:
    landscape_files = {
        "GB1": "GB1_landscape_array.pkl",
        "TrpB": "TrpB_landscape_array.pkl",
        "TEV": "TEV_landscape_array.pkl",
        "ParD3": "E3_landscape_array.pkl",
    }
    analytical_data: dict[str, object] = {}
    for landscape_name, filename in landscape_files.items():
        with (REPO_ROOT / "landscape_arrays" / filename).open("rb") as handle:
            amino_acid_landscape = np.asarray(pickle.load(handle))
        nucleotide_landscape = build_nucleotide_landscape(amino_acid_landscape)
        analytical_data[landscape_name] = {
            model: analytical_squared_decay(
                nucleotide_landscape,
                MUTATION_KERNELS[model],
                directed=model == "nuc_e_coli_directed",
            )
            for model in MODEL_KEYS
        }
    figure6_analytics = {
        "data": analytical_data,
        "params": {
            "models": MODEL_KEYS,
            "kernels": MUTATION_KERNELS,
            "normalization": "sum(I-T_i) / N_nucleotide",
        },
        "metadata": {
            "paper_reference": "Figure 6D-F and Figure S3",
            "weighted_kernel": "symmetric Sinkhorn scaling of transpose-averaged E. coli",
            "directed_asymptote": "product stationary distribution of directed E. coli kernel",
        },
    }
    save_pickle(figure6_analytics, ANALYTICS_PATH)


## Processed Sampling Accuracy for D-F


In [ ]:
def process_df_payload(raw_by_model: dict[str, list[np.ndarray]]) -> dict[str, object]:
    """Bootstrap fitted squared-decay rates for panels D-F.

    Parameters:
    - raw_by_model: dict[str, list[np.ndarray]]
        Raw all-start arrays ordered by empirical landscape.

    Returns:
    - dict[str, object]
        Bootstrap distributions and model metadata.
    """
    rng = np.random.default_rng(RANDOM_SEED)
    maxima = (160_000, 160_000, 160_000, 8_000)
    processed: dict[str, object] = {}
    for model, landscape_arrays in raw_by_model.items():
        model_results = []
        for raw, maximum in zip(landscape_arrays, maxima, strict=True):
            start_curves = np.asarray(raw, dtype=float).mean(axis=2).reshape(-1, 75)
            sample_counts = np.round(np.logspace(0, np.log10(maximum), 11)).astype(int)
            landscape_results = []
            for count in sample_counts:
                estimates = []
                for _ in range(1000):
                    selected = rng.choice(start_curves.shape[0], size=int(count), replace=True)
                    curve = np.square(start_curves[selected]).mean(axis=0)
                    curve /= max(float(curve[0]), 1e-10)
                    estimates.append(float(get_single_decay_rate_IK_v2(
                        curve, mut=DF_TOTAL_MUTATION_RATE, num_steps=DF_NUM_GENERATIONS
                    )[0] / 2.0))
                landscape_results.append(np.asarray(estimates))
            model_results.append(landscape_results)
        processed[model] = model_results
    return {
        "data": processed,
        "params": {"models": MODEL_KEYS, "M": 75, "bootstrap_replicates": 1000,
                   "seed": RANDOM_SEED, "kernels": MUTATION_KERNELS},
        "metadata": {"paper_reference": "Figure 6D-F"},
    }


if DF_PROCESSED_PATH.exists() and (PLOT_ONLY or not OVERWRITE_PROCESSED_PKL):
    figure6_df_payload = load_pickle(DF_PROCESSED_PATH)
elif PLOT_ONLY:
    raise FileNotFoundError(f"PLOT_ONLY=True requires {DF_PROCESSED_PATH}")
elif missing_df_raw_paths:
    listing = "\n".join(f"  - {path}" for path in missing_df_raw_paths)
    raise FileNotFoundError(f"Missing Figure 6 D-F raw products:\n{listing}")
else:
    raw_by_model: dict[str, list[np.ndarray]] = {}
    for model, paths in required_df_raw_paths.items():
        raw_by_model[model] = []
        for path in paths:
            with path.open("rb") as handle:
                raw_by_model[model].append(np.asarray(pickle.load(handle)))
    figure6_df_payload = process_df_payload(raw_by_model)
    save_pickle(figure6_df_payload, DF_PROCESSED_PATH)


## New Raw NK Landscapes

In [ ]:
def generate_figure6_new_landscapes(
    n_sites: int,
    num_alleles: int,
    k_values: tuple[int, ...],
    num_landscapes: int,
    seed: int,
) -> dict[str, object]:
    """Generate paired NK landscapes for the lethal and neutral analyses.

    Parameters:
    - n_sites: int
        Number of genotype sites.
    - num_alleles: int
        Number of alleles per site.
    - k_values: tuple[int, ...]
        NK epistatic interaction values.
    - num_landscapes: int
        Independent landscapes generated for every K value.
    - seed: int
        Base JAX random seed.

    Returns:
    - dict[str, object]
        Raw landscape array and complete generation metadata.
    """
    shape = (num_alleles,) * n_sites
    landscapes = np.empty(
        (len(k_values), num_landscapes, *shape),
        dtype=np.float32,
    )
    base_key = jr.PRNGKey(seed)
    for k_index, k_value in enumerate(k_values):
        for replicate in range(num_landscapes):
            landscape_key = jr.fold_in(jr.fold_in(base_key, k_value), replicate)
            landscapes[k_index, replicate] = np.asarray(
                get_nk_l_o_shape(landscape_key, n_sites, k_value, shape),
                dtype=np.float32,
            )
    return {
        "data": {"landscapes": landscapes},
        "params": {
            "N": n_sites,
            "A": num_alleles,
            "K_values": k_values,
            "num_landscapes": num_landscapes,
            "seed": seed,
        },
        "metadata": {
            "paper_reference": "Figure 6 new",
            "description": "Paired N=4, A=20 NK landscapes for lethal-node and neutral-ridge analyses.",
        },
    }


if RAW_PATH.exists() and (PLOT_ONLY or not OVERWRITE_RAW_PKL):
    figure6_new_raw = load_pickle(RAW_PATH)
elif PLOT_ONLY:
    raise FileNotFoundError(f"PLOT_ONLY=True requires raw payload {RAW_PATH}")
else:
    figure6_new_raw = generate_figure6_new_landscapes(
        N_SITES,
        NUM_ALLELES,
        K_VALUES,
        NUM_LANDSCAPES,
        RANDOM_SEED,
    )
    save_pickle(figure6_new_raw, RAW_PATH)

print(f"Raw NK landscape shape: {np.asarray(figure6_new_raw['data']['landscapes']).shape}")


## New Processed Analytical $\rho_2$

In [ ]:
def process_figure6_new_landscapes(
    raw_payload: dict[str, object],
    fractions: tuple[float, ...],
    perturbation_seed: int,
) -> dict[str, object]:
    """Compute analytical rho_2 after nested lethal and neutral perturbations.

    Parameters:
    - raw_payload: dict[str, object]
        Paired NK landscapes and generation parameters.
    - fractions: tuple[float, ...]
        Fractions of flattened values overwritten at each perturbation level.
    - perturbation_seed: int
        Seed controlling node order, ridge site, reference allele, and allele order.

    Returns:
    - dict[str, object]
        Replicate-level rho_2 values, summaries, perturbation choices, and metadata.
    """
    landscapes = np.asarray(raw_payload["data"]["landscapes"], dtype=np.float32)
    params = raw_payload["params"]
    n_sites = int(params["N"])
    num_alleles = int(params["A"])
    k_values = tuple(int(value) for value in params["K_values"])
    num_landscapes = int(params["num_landscapes"])
    fraction_array = np.asarray(fractions, dtype=float)
    if np.any(np.diff(fraction_array) < 0) or fraction_array[0] != 0.0:
        raise ValueError("Fractions must be increasing and begin at zero.")

    expected_shape = (len(k_values), num_landscapes, *((num_alleles,) * n_sites))
    if landscapes.shape != expected_shape:
        raise ValueError(f"Expected landscape shape {expected_shape}, received {landscapes.shape}.")

    output_shape = (len(k_values), len(fractions), num_landscapes)
    lethal_rho2 = np.empty(output_shape, dtype=float)
    neutral_rho2 = np.empty(output_shape, dtype=float)
    ridge_sites = np.empty((len(k_values), num_landscapes), dtype=np.int16)
    reference_alleles = np.empty((len(k_values), num_landscapes), dtype=np.int16)
    alternative_allele_orders = np.empty(
        (len(k_values), num_landscapes, num_alleles - 1),
        dtype=np.int16,
    )
    total_nodes = num_alleles ** n_sites
    lethal_counts = np.rint(fraction_array * total_nodes).astype(int)
    neutral_counts = np.rint(fraction_array * num_alleles).astype(int)
    if np.any(neutral_counts > num_alleles - 1):
        raise ValueError("Neutral fractions request more alternative alleles than are available.")

    for k_index, k_value in enumerate(k_values):
        for replicate in range(num_landscapes):
            landscape = landscapes[k_index, replicate]
            rng = np.random.default_rng(perturbation_seed + 10_000 * k_value + replicate)
            lethal_order = rng.permutation(total_nodes)
            ridge_site = int(rng.integers(0, n_sites))
            reference_allele = int(rng.integers(0, num_alleles))
            alternative_alleles = np.delete(np.arange(num_alleles), reference_allele)
            rng.shuffle(alternative_alleles)
            ridge_sites[k_index, replicate] = ridge_site
            reference_alleles[k_index, replicate] = reference_allele
            alternative_allele_orders[k_index, replicate] = alternative_alleles

            base_rho2 = float(get_dirichlet_metric(landscape))
            lethal_rho2[k_index, 0, replicate] = base_rho2
            neutral_rho2[k_index, 0, replicate] = base_rho2
            landscape_minimum = float(landscape.min())
            reference_index = [slice(None)] * n_sites
            reference_index[ridge_site] = reference_allele
            reference_slice = landscape[tuple(reference_index)].copy()

            for fraction_index in range(1, len(fractions)):
                lethal_landscape = landscape.copy()
                lethal_landscape.reshape(-1)[
                    lethal_order[:lethal_counts[fraction_index]]
                ] = landscape_minimum
                lethal_rho2[k_index, fraction_index, replicate] = float(
                    get_dirichlet_metric(lethal_landscape)
                )

                neutral_landscape = landscape.copy()
                for allele in alternative_alleles[:neutral_counts[fraction_index]]:
                    target_index = [slice(None)] * n_sites
                    target_index[ridge_site] = int(allele)
                    neutral_landscape[tuple(target_index)] = reference_slice
                    if not np.array_equal(neutral_landscape[tuple(target_index)], reference_slice):
                        raise AssertionError("Neutral allele slice was not copied exactly.")
                neutral_rho2[k_index, fraction_index, replicate] = float(
                    get_dirichlet_metric(neutral_landscape)
                )

    if lethal_rho2.shape != output_shape or neutral_rho2.shape != output_shape:
        raise AssertionError("Processed rho_2 arrays have an unexpected shape.")
    if not np.array_equal(lethal_rho2[:, 0, :], neutral_rho2[:, 0, :]):
        raise AssertionError("Zero-fraction lethal and neutral rho_2 values differ.")
    if not np.all(np.isfinite(lethal_rho2)) or not np.all(np.isfinite(neutral_rho2)):
        raise FloatingPointError("Processed rho_2 arrays contain non-finite values.")
    if np.any(np.diff(lethal_counts) < 0) or np.any(np.diff(neutral_counts) < 0):
        raise AssertionError("Perturbation sets are not nested.")

    return {
        "data": {
            "fractions": fraction_array,
            "lethal_rho2": lethal_rho2,
            "neutral_rho2": neutral_rho2,
            "lethal_mean": lethal_rho2.mean(axis=2),
            "lethal_std": lethal_rho2.std(axis=2, ddof=1),
            "neutral_mean": neutral_rho2.mean(axis=2),
            "neutral_std": neutral_rho2.std(axis=2, ddof=1),
            "ridge_sites": ridge_sites,
            "reference_alleles": reference_alleles,
            "alternative_allele_orders": alternative_allele_orders,
            "lethal_counts": lethal_counts,
            "neutral_counts": neutral_counts,
        },
        "params": {
            "N": n_sites,
            "A": num_alleles,
            "K_values": k_values,
            "num_landscapes": num_landscapes,
            "fractions": fractions,
            "perturbation_seed": perturbation_seed,
        },
        "metadata": {
            "paper_reference": "Figure 6 new panels G-H",
            "standard_deviation_ddof": 1,
            "lethal_definition": "Nested random nodes forced to each landscape's global minimum.",
            "neutral_definition": "Nested alternative-allele slices copied from one reference allele at one focal site.",
            "neutral_fraction_definition": "Fraction of genotype values overwritten.",
        },
    }


if PROCESSED_PATH.exists() and (PLOT_ONLY or not OVERWRITE_PROCESSED_PKL):
    figure6_new_payload = load_pickle(PROCESSED_PATH)
elif PLOT_ONLY:
    raise FileNotFoundError(f"PLOT_ONLY=True requires processed payload {PROCESSED_PATH}")
else:
    figure6_new_payload = process_figure6_new_landscapes(
        figure6_new_raw,
        FRACTIONS,
        RANDOM_SEED + 1,
    )
    save_pickle(figure6_new_payload, PROCESSED_PATH)

for key in ("lethal_rho2", "neutral_rho2"):
    values = np.asarray(figure6_new_payload["data"][key])
    assert values.shape == (4, 6, 20)
    assert np.all(np.isfinite(values))
assert np.array_equal(
    np.asarray(figure6_new_payload["data"]["lethal_rho2"])[:, 0, :],
    np.asarray(figure6_new_payload["data"]["neutral_rho2"])[:, 0, :],
)
assert int(figure6_new_payload["metadata"]["standard_deviation_ddof"]) == 1
print("Validated processed lethal and neutral rho_2 payloads.")


## Raw Fitness-Decay Trajectories for G--H

Generate and save every population-replicate trajectory used to estimate fitted $\rho_2$.

In [ ]:
def reconstruct_perturbed_landscape(
    raw_payload: dict[str, object],
    analytical_payload: dict[str, object],
    perturbation: str,
    k_index: int,
    fraction_index: int,
    landscape_index: int,
) -> np.ndarray:
    """Reconstruct a panel G/H perturbed landscape.

    Parameters:
    - raw_payload: dict[str, object]
        Original NK landscapes.
    - analytical_payload: dict[str, object]
        Existing analytical perturbation results.
    - perturbation: str
        Lethal or neutral perturbation name.
    - k_index: int
        K-value index.
    - fraction_index: int
        Perturbation-fraction index.
    - landscape_index: int
        Landscape-replicate index.

    Returns:
    - np.ndarray
        Reconstructed perturbed landscape.
    """
    if perturbation not in PERTURBATIONS:
        raise ValueError(f"Unknown perturbation {perturbation!r}.")
    landscape = np.asarray(raw_payload["data"]["landscapes"], dtype=np.float32)[
        k_index, landscape_index
    ]
    data = analytical_payload["data"]
    params = analytical_payload["params"]
    k_value = int(params["K_values"][k_index])
    rng = np.random.default_rng(
        int(params["perturbation_seed"]) + 10_000 * k_value + landscape_index
    )
    lethal_order = rng.permutation(landscape.size)
    ridge_site = int(rng.integers(0, landscape.ndim))
    reference_allele = int(rng.integers(0, landscape.shape[ridge_site]))
    alternative_alleles = np.delete(
        np.arange(landscape.shape[ridge_site]), reference_allele
    )
    rng.shuffle(alternative_alleles)
    if ridge_site != int(np.asarray(data["ridge_sites"])[k_index, landscape_index]):
        raise AssertionError("Reconstructed ridge site differs from analytical metadata.")
    if reference_allele != int(
        np.asarray(data["reference_alleles"])[k_index, landscape_index]
    ):
        raise AssertionError("Reconstructed reference allele differs from metadata.")
    if not np.array_equal(
        alternative_alleles,
        np.asarray(data["alternative_allele_orders"])[k_index, landscape_index],
    ):
        raise AssertionError("Reconstructed allele order differs from metadata.")

    perturbed = landscape.copy()
    if perturbation == "lethal":
        count = int(np.asarray(data["lethal_counts"])[fraction_index])
        perturbed.reshape(-1)[lethal_order[:count]] = float(landscape.min())
    else:
        count = int(np.asarray(data["neutral_counts"])[fraction_index])
        reference_index = [slice(None)] * landscape.ndim
        reference_index[ridge_site] = reference_allele
        reference_slice = landscape[tuple(reference_index)].copy()
        for allele in alternative_alleles[:count]:
            target_index = [slice(None)] * landscape.ndim
            target_index[ridge_site] = int(allele)
            perturbed[tuple(target_index)] = reference_slice
    return perturbed


def generate_figure6_decay_raw(
    raw_payload: dict[str, object],
    analytical_payload: dict[str, object],
    seed: int,
) -> dict[str, object]:
    """Generate all replicate fitness-decay trajectories for panels G/H.

    Parameters:
    - raw_payload: dict[str, object]
        Original NK landscapes.
    - analytical_payload: dict[str, object]
        Existing analytical perturbation results.
    - seed: int
        Master selection and simulation seed.

    Returns:
    - dict[str, object]
        Trajectories, starts, seeds, parameters, and metadata.
    """
    analytical_data = analytical_payload["data"]
    analytical_params = analytical_payload["params"]
    fractions = np.asarray(analytical_data["fractions"], dtype=float)
    k_values = tuple(int(value) for value in analytical_params["K_values"])
    num_landscapes = int(analytical_params["num_landscapes"])
    trajectory_shape = (
        len(PERTURBATIONS), len(k_values), len(fractions), num_landscapes,
        len(START_CLASSES), NUM_STARTS_PER_CLASS,
        NUM_POPULATION_REPLICATES, NUM_DECAY_STEPS,
    )
    trajectories = np.empty(trajectory_shape, dtype=np.float32)
    start_coordinates = np.empty(trajectory_shape[:6] + (N_SITES,), dtype=np.int16)
    initial_fitness = np.empty(trajectory_shape[:6], dtype=np.float32)
    selection_seeds = np.empty(trajectory_shape[:4], dtype=np.uint32)
    simulation_seeds = np.empty(trajectory_shape[:4], dtype=np.uint32)

    for perturbation_index, perturbation in enumerate(PERTURBATIONS):
        for k_index, k_value in enumerate(k_values):
            for fraction_index in range(len(fractions)):
                for landscape_index in range(num_landscapes):
                    perturbed = reconstruct_perturbed_landscape(
                        raw_payload, analytical_payload, perturbation,
                        k_index, fraction_index, landscape_index,
                    )
                    expected_rho2 = float(
                        np.asarray(analytical_data[f"{perturbation}_rho2"])[
                            k_index, fraction_index, landscape_index
                        ]
                    )
                    if not np.isclose(
                        float(get_dirichlet_metric(perturbed)),
                        expected_rho2,
                        rtol=1e-5,
                        atol=1e-6,
                    ):
                        raise AssertionError(
                            "Reconstructed perturbation differs from analytical result."
                        )
                    stream_index = (
                        ((perturbation_index * len(k_values) + k_index)
                         * len(fractions) + fraction_index)
                        * num_landscapes + landscape_index
                    )
                    selection_seed = seed + 2 * stream_index
                    simulation_seed = selection_seed + 1
                    output_index = (
                        perturbation_index, k_index,
                        fraction_index, landscape_index,
                    )
                    selection_seeds[output_index] = selection_seed
                    simulation_seeds[output_index] = simulation_seed
                    selection_rng = np.random.default_rng(selection_seed)
                    flat_fitness = perturbed.reshape(-1)
                    mean_distance = np.abs(flat_fitness - flat_fitness.mean())
                    near_candidates = np.flatnonzero(
                        mean_distance <= np.quantile(mean_distance, 0.25)
                    )
                    high_candidates = np.flatnonzero(
                        flat_fitness >= np.quantile(flat_fitness, 0.80)
                    )
                    near_flat = selection_rng.choice(
                        near_candidates, size=NUM_STARTS_PER_CLASS, replace=False
                    )
                    high_candidates = np.setdiff1d(
                        high_candidates, near_flat, assume_unique=False
                    )
                    high_flat = selection_rng.choice(
                        high_candidates, size=NUM_STARTS_PER_CLASS, replace=False
                    )
                    if np.intersect1d(near_flat, high_flat).size:
                        raise AssertionError("Start classes overlap.")
                    selected_flat = np.stack((near_flat, high_flat))
                    selected_coordinates = np.stack([
                        np.column_stack(np.unravel_index(group, perturbed.shape))
                        for group in selected_flat
                    ]).astype(np.int16)
                    start_coordinates[output_index] = selected_coordinates
                    initial_fitness[output_index] = flat_fitness[selected_flat]
                    simulated = generate_empirical_decay_curves(
                        perturbed,
                        mutation_rate=TOTAL_MUTATION_RATE / N_SITES,
                        popsize=POPULATION_SIZE,
                        starts=selected_coordinates.reshape(-1, N_SITES),
                        num_reps=NUM_POPULATION_REPLICATES,
                        num_steps=NUM_DECAY_STEPS,
                        seed=simulation_seed,
                        batch_size=2 * NUM_STARTS_PER_CLASS,
                    )
                    trajectories[output_index] = simulated.reshape(
                        len(START_CLASSES), NUM_STARTS_PER_CLASS,
                        NUM_POPULATION_REPLICATES, NUM_DECAY_STEPS,
                    )

    if not np.all(np.isfinite(trajectories)):
        raise AssertionError("Raw decay trajectories contain non-finite values.")
    return {
        "data": {
            "fitness_trajectories": trajectories,
            "start_coordinates": start_coordinates,
            "initial_fitness": initial_fitness,
            "selection_seeds": selection_seeds,
            "simulation_seeds": simulation_seeds,
        },
        "params": {
            "N": N_SITES,
            "A": NUM_ALLELES,
            "K_values": k_values,
            "fractions": tuple(float(value) for value in fractions),
            "num_landscapes": num_landscapes,
            "perturbations": PERTURBATIONS,
            "start_classes": START_CLASSES,
            "num_starts_per_class": NUM_STARTS_PER_CLASS,
            "num_population_replicates": NUM_POPULATION_REPLICATES,
            "population_size": POPULATION_SIZE,
            "total_mutation_rate": TOTAL_MUTATION_RATE,
            "mutation_rate_per_site": TOTAL_MUTATION_RATE / N_SITES,
            "num_decay_steps": NUM_DECAY_STEPS,
            "seed": seed,
        },
        "metadata": {
            "paper_reference": "Figure 6 new panels G-H fitted decay rates",
            "near_average_definition": (
                "Random sample from the 25% closest to the complete "
                "perturbed-landscape mean."
            ),
            "high_fitness_definition": (
                "Random sample from the top 20% of the complete perturbed landscape."
            ),
            "trajectory_axes": (
                "perturbation", "K", "fraction", "landscape", "start_class",
                "start", "population_replicate", "generation",
            ),
        },
    }


if DECAY_RAW_PATH.exists() and (PLOT_ONLY or not OVERWRITE_RAW_PKL):
    figure6_decay_raw = load_pickle(DECAY_RAW_PATH)
elif PLOT_ONLY:
    if DECAY_PROCESSED_PATH.exists():
        figure6_decay_raw = None
    else:
        raise FileNotFoundError(
            f"PLOT_ONLY=True requires {DECAY_PROCESSED_PATH} or {DECAY_RAW_PATH}"
        )
else:
    figure6_decay_raw = generate_figure6_decay_raw(
        figure6_new_raw, figure6_new_payload, RANDOM_SEED + 2
    )
    save_pickle(figure6_decay_raw, DECAY_RAW_PATH)

if figure6_decay_raw is not None:
    expected_shape = (2, 4, 6, 20, 2, 10, 10, 75)
    raw_trajectories = np.asarray(
        figure6_decay_raw["data"]["fitness_trajectories"]
    )
    assert raw_trajectories.shape == expected_shape
    assert np.all(np.isfinite(raw_trajectories))
    raw_starts = np.asarray(figure6_decay_raw["data"]["start_coordinates"])
    assert raw_starts.shape == expected_shape[:6] + (N_SITES,)
    assert np.all(raw_starts >= 0) and np.all(raw_starts < NUM_ALLELES)
    print("Validated raw Figure 6 G-H decay payload.")


## Processed Fitted $\rho_2$ for G--H

Average population replicates per start, square each $F_\mu$, and fit local and pooled decay rates.

In [ ]:
def process_figure6_decay_raw(
    decay_raw_payload: dict[str, object],
) -> dict[str, object]:
    """Fit local and pooled rho_2 values from raw fitness trajectories.

    Parameters:
    - decay_raw_payload: dict[str, object]
        Replicate-level fitness trajectories and simulation parameters.

    Returns:
    - dict[str, object]
        Fit results, failures, summaries, parameters, and metadata.
    """
    trajectories = np.asarray(
        decay_raw_payload["data"]["fitness_trajectories"], dtype=float
    )
    params = decay_raw_payload["params"]
    mutation_scale = 2.0 * float(params["total_mutation_rate"])
    num_steps = int(params["num_decay_steps"])
    f_mu = trajectories.mean(axis=6)
    g_mu_local = np.square(f_mu)
    local_shape = g_mu_local.shape[:-1]
    local_rho2 = np.full(local_shape, np.nan, dtype=float)
    local_asymptote = np.full(local_shape, np.nan, dtype=float)
    local_fit_success = np.zeros(local_shape, dtype=bool)
    pooled_shape = local_shape[:4]
    pooled_rho2 = np.full(pooled_shape, np.nan, dtype=float)
    pooled_asymptote = np.full(pooled_shape, np.nan, dtype=float)
    pooled_fit_success = np.zeros(pooled_shape, dtype=bool)
    fit_failures: list[dict[str, object]] = []

    for index in np.ndindex(local_shape):
        curve = g_mu_local[index]
        try:
            if not np.all(np.isfinite(curve)) or np.isclose(curve[0], 0.0):
                raise ValueError("Local G_mu is non-finite or begins at zero.")
            rate, asymptote = get_single_decay_rate(
                curve, mut=mutation_scale, num_steps=num_steps
            )
            local_rho2[index] = float(rate)
            local_asymptote[index] = float(asymptote)
            local_fit_success[index] = True
        except (RuntimeError, ValueError, FloatingPointError) as error:
            fit_failures.append({
                "fit_type": "local",
                "index": tuple(int(value) for value in index),
                "error": str(error),
            })

    for index in np.ndindex(pooled_shape):
        pooled_curve = g_mu_local[index].mean(axis=(0, 1))
        try:
            if not np.all(np.isfinite(pooled_curve)) or np.isclose(
                pooled_curve[0], 0.0
            ):
                raise ValueError("Pooled G_mu is non-finite or begins at zero.")
            rate, asymptote = get_single_decay_rate(
                pooled_curve, mut=mutation_scale, num_steps=num_steps
            )
            pooled_rho2[index] = float(rate)
            pooled_asymptote[index] = float(asymptote)
            pooled_fit_success[index] = True
        except (RuntimeError, ValueError, FloatingPointError) as error:
            fit_failures.append({
                "fit_type": "pooled",
                "index": tuple(int(value) for value in index),
                "error": str(error),
            })

    class_rho2_by_landscape = np.nanmean(local_rho2, axis=5)
    class_mean = np.nanmean(class_rho2_by_landscape, axis=3)
    class_std = np.nanstd(class_rho2_by_landscape, axis=3, ddof=1)
    pooled_mean = np.nanmean(pooled_rho2, axis=3)
    pooled_std = np.nanstd(pooled_rho2, axis=3, ddof=1)
    if class_mean.shape != (2, 4, 6, 2):
        raise AssertionError("Class fitted-rho summaries have unexpected dimensions.")
    if pooled_mean.shape != (2, 4, 6):
        raise AssertionError("Pooled fitted-rho summaries have unexpected dimensions.")
    return {
        "data": {
            "f_mu": f_mu,
            "g_mu_local": g_mu_local,
            "local_rho2": local_rho2,
            "local_asymptote": local_asymptote,
            "local_fit_success": local_fit_success,
            "class_rho2_by_landscape": class_rho2_by_landscape,
            "class_mean": class_mean,
            "class_std": class_std,
            "pooled_rho2": pooled_rho2,
            "pooled_asymptote": pooled_asymptote,
            "pooled_fit_success": pooled_fit_success,
            "pooled_mean": pooled_mean,
            "pooled_std": pooled_std,
            "fit_failures": fit_failures,
        },
        "params": dict(params),
        "metadata": {
            "paper_reference": "Figure 6 new panels G-H fitted decay rates",
            "fit_model": (
                "Single exponential fitted to G_mu with mutation scale "
                "2 * total mutation rate."
            ),
            "local_summary": (
                "Ten local fits averaged per class and landscape; "
                "mean and sample SD across landscapes."
            ),
            "pooled_summary": (
                "One fit to G_mu pooled over all 20 starts per landscape; "
                "mean and sample SD across landscapes."
            ),
            "standard_deviation_ddof": 1,
            "num_fit_failures": len(fit_failures),
        },
    }


if DECAY_PROCESSED_PATH.exists() and (
    PLOT_ONLY or not OVERWRITE_PROCESSED_PKL
):
    figure6_decay_processed = load_pickle(DECAY_PROCESSED_PATH)
elif PLOT_ONLY:
    raise FileNotFoundError(
        f"PLOT_ONLY=True requires processed payload {DECAY_PROCESSED_PATH}"
    )
else:
    if figure6_decay_raw is None:
        raise RuntimeError("Raw decay trajectories are required for processing.")
    figure6_decay_processed = process_figure6_decay_raw(figure6_decay_raw)
    save_pickle(figure6_decay_processed, DECAY_PROCESSED_PATH)

decay_data = figure6_decay_processed["data"]
assert np.asarray(decay_data["local_rho2"]).shape == (2, 4, 6, 20, 2, 10)
assert np.asarray(decay_data["pooled_rho2"]).shape == (2, 4, 6, 20)
assert np.asarray(decay_data["class_mean"]).shape == (2, 4, 6, 2)
assert np.asarray(decay_data["pooled_mean"]).shape == (2, 4, 6)
assert int(figure6_decay_processed["metadata"]["standard_deviation_ddof"]) == 1
print(
    "Validated processed Figure 6 G-H decay payload; "
    f"{figure6_decay_processed['metadata']['num_fit_failures']} fits failed."
)


## Plotting Functions


In [ ]:
def plot_abc_panel(ax: Axes, payload: dict[str, object], model: str) -> None:
    """Plot one synthetic-NK local squared-decay panel.

    Parameters:
    - ax: Axes
        Axis receiving the plot.
    - payload: dict[str, object]
        Processed A-C payload.
    - model: str
        Mutation-model key.

    Returns:
    - None
        Artists are added to the axis.
    """
    panel = payload["data"][model]
    rho = np.asarray(panel["binned_rho_NK"])
    means = np.asarray(panel["binned_mean"])
    stds = np.asarray(panel["binned_std"])
    symbol = r"$\rho_2^{\mathrm{loc}}$" if model == "nuc_uniform" else (
        r"$\widetilde{\rho}_2^{\mathrm{loc}}$" if model == "nuc_e_coli_weighted"
        else r"$\overline{\rho}_2^{\mathrm{loc}}$"
    )
    ax.plot(rho, means, "o-", linewidth=1.4, markersize=4, label=symbol)
    ax.fill_between(rho, means - stds, means + stds, alpha=0.25, linewidth=0)
    ax.plot((0, 1), (0, 1), color="red", linestyle="--", alpha=0.55, label=r"$\rho_{NK}$")
    ax.set_title(MODEL_TITLES[model], fontsize=FIGURE_TITLE_SIZE)
    ax.set_xlabel(r"$\rho_{NK}$", fontsize=FIGURE_LABEL_SIZE)
    ax.set_ylabel(symbol, fontsize=FIGURE_LABEL_SIZE)
    ax.set_xlim(0, 1.02)
    ax.set_ylim(0, 1.10)
    ax.legend(fontsize=FIGURE_LEGEND_SIZE, loc="upper left")
    ax.tick_params(labelsize=FIGURE_TICK_SIZE)
    ax.grid(True, alpha=0.18)


def plot_df_panel(ax: Axes, payload: dict[str, object], model: str) -> None:
    """Plot sampling accuracy and its analytical reference for one kernel.

    Parameters:
    - ax: Axes
        Axis receiving the plot.
    - payload: dict[str, object]
        Processed D-F bootstrap payload.
    - model: str
        Mutation-model key.

    Returns:
    - None
        Artists are added to the axis.
    """
    maxima = (160_000, 160_000, 160_000, 8_000)
    symbol = r"$\rho_2^{\mathrm{fit}}$" if model == "nuc_uniform" else (
        r"$\widetilde{\rho}_2^{\mathrm{fit}}$" if model == "nuc_e_coli_weighted"
        else r"$\overline{\rho}_2^{\mathrm{fit}}$"
    )
    reference_symbol = r"$\rho_2$" if model == "nuc_uniform" else (
        r"$\widetilde{\rho}_2$" if model == "nuc_e_coli_weighted" else r"$\overline{\rho}_2$"
    )
    for index, (name, maximum, color, marker) in enumerate(zip(
        LANDSCAPE_NAMES, maxima, LANDSCAPE_COLORS, LANDSCAPE_MARKERS, strict=True
    )):
        starts = np.round(np.logspace(0, np.log10(maximum), 11)).astype(int)
        values = payload["data"][model][index]
        means = np.asarray([np.mean(item) for item in values])
        stds = np.asarray([np.std(item) for item in values])
        ax.plot(starts, means, color=color, marker=marker, markersize=3.5,
                markevery=2, linewidth=1.3, label=name)
        ax.fill_between(starts, means - stds, means + stds, color=color, alpha=0.15)
        reference = float(figure6_analytics["data"][name][model]["rho_2"])
        ax.axhline(reference, color=color, linestyle="--", linewidth=1.0, alpha=0.65)
    ax.set_xscale("log")
    ax.set_title(MODEL_TITLES[model], fontsize=FIGURE_TITLE_SIZE)
    ax.set_xlabel("Number of starting points", fontsize=FIGURE_LABEL_SIZE)
    ax.set_ylabel(symbol, fontsize=FIGURE_LABEL_SIZE)
    ax.set_xlim(1, 160_000)
    ax.set_ylim(0, 2)
    handles, labels = ax.get_legend_handles_labels()
    handles.append(mlines.Line2D([], [], color="black", linestyle="--", label=reference_symbol))
    labels.append(reference_symbol)
    ax.legend(handles, labels, fontsize=FIGURE_LEGEND_SIZE, loc="upper right")
    ax.tick_params(labelsize=FIGURE_TICK_SIZE)


## Individual Panels A-C


In [ ]:
for letter, model in zip("ABC", MODEL_KEYS, strict=True):
    fig, ax = plt.subplots(figsize=(3.2, 2.8), dpi=PANEL_DPI)
    plot_abc_panel(ax, figure6_abc_payload, model)
    add_panel_letter(ax, letter)
    if SAVE_FIGURES:
        save_figure(fig, f"figure_6_new{letter}")
    plt.show()


## Individual Panels D-F


In [ ]:
for letter, model in zip("DEF", MODEL_KEYS, strict=True):
    fig, ax = plt.subplots(figsize=(3.2, 2.8), dpi=PANEL_DPI)
    plot_df_panel(ax, figure6_df_payload, model)
    add_panel_letter(ax, letter)
    if SAVE_FIGURES:
        save_figure(fig, f"figure_6_new{letter}")
    plt.show()


## Individual Panels G--H

In [ ]:

def plot_figure6_new_rho2_panel(
    ax: Axes,
    payload: dict[str, object],
    decay_payload: dict[str, object],
    perturbation: str,
) -> None:
    """Plot analytical rho_2 against a controlled landscape perturbation.

    Parameters:
    - ax: Axes
        Axis receiving the plot.
    - payload: dict[str, object]
        Processed lethal-node and neutral-ridge results.
    - decay_payload: dict[str, object]
        Processed local and pooled fitted decay rates.
    - perturbation: str
        Either ``lethal`` or ``neutral``.

    Returns:
    - None
        Plot artists are added directly to ``ax``.
    """
    if perturbation not in {"lethal", "neutral"}:
        raise ValueError("perturbation must be 'lethal' or 'neutral'.")
    data = payload["data"]
    params = payload["params"]
    fractions = np.asarray(data["fractions"], dtype=float)
    means = np.asarray(data[f"{perturbation}_mean"], dtype=float)
    standard_deviations = np.asarray(data[f"{perturbation}_std"], dtype=float)
    decay_data = decay_payload["data"]
    perturbation_index = PERTURBATIONS.index(perturbation)
    class_means = np.asarray(decay_data["class_mean"], dtype=float)[perturbation_index]
    class_stds = np.asarray(decay_data["class_std"], dtype=float)[perturbation_index]
    pooled_means = np.asarray(decay_data["pooled_mean"], dtype=float)[perturbation_index]
    pooled_stds = np.asarray(decay_data["pooled_std"], dtype=float)[perturbation_index]
    k_values = tuple(int(value) for value in params["K_values"])
    n_sites = int(params["N"])
    colors = ("tab:blue", "tab:orange", "tab:green", "tab:red")
    markers = ("o", "s", "D", "^")

    for k_index, k_value in enumerate(k_values):
        color = colors[k_index]
        ax.plot(
            fractions,
            means[k_index],
            color=color,
            marker=markers[k_index],
            linewidth=1.4,
            markersize=4,
            label=rf"$K={k_value}$",
        )
        ax.fill_between(
            fractions,
            means[k_index] - standard_deviations[k_index],
            means[k_index] + standard_deviations[k_index],
            color=color,
            alpha=0.16,
            linewidth=0,
        )
        fitted_series = (
            (class_means[k_index, :, 0], class_stds[k_index, :, 0], "--"),
            (class_means[k_index, :, 1], class_stds[k_index, :, 1], "-."),
            (pooled_means[k_index], pooled_stds[k_index], (0, (3, 1, 1, 1))),
        )
        for fitted_mean, fitted_std, line_style in fitted_series:
            ax.plot(
                fractions, fitted_mean, color=color, marker=markers[k_index],
                linestyle=line_style, linewidth=1.15, markersize=3.2,
            )
            ax.fill_between(
                fractions, fitted_mean - fitted_std, fitted_mean + fitted_std,
                color=color, alpha=0.07, linewidth=0,
            )
        ax.axhline(
            (k_value + 1) / n_sites,
            color=color,
            linestyle=":",
            linewidth=1.0,
            alpha=0.9,
        )

    k_handles = [
        mlines.Line2D([], [], color=colors[index], marker=markers[index],
                      linestyle="-", label=rf"$K={k_value}$")
        for index, k_value in enumerate(k_values)
    ]
    estimator_handles = [
        mlines.Line2D([], [], color="black", linestyle="-", label=r"Analytical $\rho_2$"),
        mlines.Line2D([], [], color="black", linestyle="--", label=r"Near-average $\rho_2^{\mathrm{loc}}$"),
        mlines.Line2D([], [], color="black", linestyle="-.", label=r"High-fitness $\rho_2^{\mathrm{loc}}$"),
        mlines.Line2D([], [], color="black", linestyle=(0, (3, 1, 1, 1)), label=r"All-start $\rho_2^{\mathrm{fit}}$"),
        mlines.Line2D([], [], color="black", linestyle=":", label=r"$\rho_{NK}$"),
    ]
    ax.legend(
        handles=k_handles + estimator_handles,
        fontsize=FIGURE_LEGEND_SIZE,
        ncol=2,
        frameon=True,
        loc="best",
    )
    if perturbation == "lethal":
        ax.set_title(rf"Minimum-fitness genotypes (N={N_SITES}, A={NUM_ALLELES})", fontsize=FIGURE_TITLE_SIZE)
        ax.set_xlabel("Fraction forced to minimum fitness", fontsize=FIGURE_LABEL_SIZE)
    else:
        ax.set_title(rf"Neutral allele ridges (N={N_SITES}, A={NUM_ALLELES})", fontsize=FIGURE_TITLE_SIZE)
        ax.set_xlabel("Fraction of genotype values overwritten", fontsize=FIGURE_LABEL_SIZE)
    ax.set_ylabel(r"$\rho_2$", fontsize=FIGURE_LABEL_SIZE)
    ax.set_xlim(-0.01, 0.51)
    upper_limit = 1.05 * max(
        1.10,
        float(np.nanmax(means + standard_deviations)),
        float(np.nanmax(class_means + class_stds)),
        float(np.nanmax(pooled_means + pooled_stds)),
    )
    ax.set_ylim(0.0, upper_limit)
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    ax.tick_params(labelsize=FIGURE_TICK_SIZE)
    ax.grid(True, alpha=0.16)

for letter, perturbation in zip("GH", ("lethal", "neutral"), strict=True):
    fig, ax = plt.subplots(figsize=(4.0, 3.0), dpi=PANEL_DPI)
    plot_figure6_new_rho2_panel(
        ax, figure6_new_payload, figure6_decay_processed, perturbation
    )
    add_panel_letter(ax, letter)
    if SAVE_FIGURES:
        save_figure(fig, f"figure_6_new{letter}")
    plt.show()


## Complete Figure 6 New A-H


In [ ]:
fig = plt.figure(figsize=(8, 7.5), dpi=PANEL_DPI, constrained_layout=True)
grid = GridSpec(3, 6, figure=fig, height_ratios=[1.0, 1.0, 1.0])

abc_axes = [fig.add_subplot(grid[0, 2 * column:2 * column + 2]) for column in range(3)]
for ax, letter, model in zip(abc_axes, "ABC", MODEL_KEYS, strict=True):
    plot_abc_panel(ax, figure6_abc_payload, model)
    add_panel_letter(ax, letter)

df_axes = [fig.add_subplot(grid[1, 2 * column:2 * column + 2]) for column in range(3)]
for ax, letter, model in zip(df_axes, "DEF", MODEL_KEYS, strict=True):
    plot_df_panel(ax, figure6_df_payload, model)
    add_panel_letter(ax, letter)

rho_axes = [fig.add_subplot(grid[2, :3]), fig.add_subplot(grid[2, 3:])]
for ax, letter, perturbation in zip(rho_axes, "GH", ("lethal", "neutral"), strict=True):
    plot_figure6_new_rho2_panel(ax, figure6_new_payload, figure6_decay_processed, perturbation)
    add_panel_letter(ax, letter)

if SAVE_FIGURES:
    save_figure(fig, "figure_6_new")
plt.show()
